In [4]:
from pathlib import Path
import sys
import os
import numpy as np


try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive") # mount the drive
    DATA_PATH = Path("/content/drive/MyDrive") # the root of the drive
    DATA_ROOT = DATA_PATH / "cbc_pe_data" # the root of the data
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))
    DATA_ROOT = PROJECT_ROOT / "data"


DATA_RAW = DATA_ROOT / "raw" # the root of the raw data
DATA_PROCESSED = DATA_ROOT / "processed" # the root of the processed data
MODELS_DIR = DATA_ROOT / "models" # the root of the models
RESULTS_DIR = DATA_ROOT / "results" # the root of the results
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints" # the root of the checkpoints

for path in [DATA_RAW, DATA_PROCESSED, MODELS_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("DATA_RAW:", DATA_RAW)
print("MODELS_DIR:", MODELS_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("CHECKPOINTS_DIR:", CHECKPOINTS_DIR)



#Clone the repository if it doesn't exist
if IN_COLAB:
    REPO_ROOT = Path("/content/Gravitational-Waves-Lab")
    PROJECT_ROOT = REPO_ROOT / "cbc_pe"

    if not REPO_ROOT.exists():
        !git clone https://github.com/victorsh13/Gravitational-Waves-Lab.git /content/Gravitational-Waves-Lab

    print("REPO_ROOT exists:", REPO_ROOT.exists())
    print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

    #Pull the latest version of the repository
    %cd /content/Gravitational-Waves-Lab
    !git pull
    !git status

    #Go to the project root
    %cd /content/Gravitational-Waves-Lab/cbc_pe
else:
    %cd /home/victor/gw/cbc_pe
    


# Add the project root to the python path
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("cwd:", Path.cwd())
print("sys.path[:3]:", sys.path[:3])

DATA_ROOT: /home/victor/gw/Gravitational-Waves-Lab/cbc_pe/data
DATA_RAW: /home/victor/gw/Gravitational-Waves-Lab/cbc_pe/data/raw
MODELS_DIR: /home/victor/gw/Gravitational-Waves-Lab/cbc_pe/data/models
RESULTS_DIR: /home/victor/gw/Gravitational-Waves-Lab/cbc_pe/data/results
CHECKPOINTS_DIR: /home/victor/gw/Gravitational-Waves-Lab/cbc_pe/data/models/checkpoints
[Errno 2] No such file or directory: '/home/victor/gw/cbc_pe'
/home/victor/gw/Gravitational-Waves-Lab/cbc_pe/notebooks
cwd: /home/victor/gw/Gravitational-Waves-Lab/cbc_pe
sys.path[:3]: ['/home/victor/gw/Gravitational-Waves-Lab/cbc_pe', '/home/victor/miniconda3/envs/gw-env/lib/python311.zip', '/home/victor/miniconda3/envs/gw-env/lib/python3.11']


In [5]:
from pathlib import Path
import numpy as np

from src.io import load_dataset_npz

dataset_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n4500"

dataset_path = DATA_PROCESSED / f"{dataset_id}.npz"
split_path = DATA_PROCESSED / f"{dataset_id}_splits.npz"

batch = load_dataset_npz(dataset_path)

X = batch.X
y = batch.y
metadata = batch.metadata
label_names = ["chirp_mass", "total_mass", "chi_eff"]


splits = np.load(split_path)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
cal_idx = splits["cal_idx"]
test_idx = splits["test_idx"]

X_train, y_train_phys = X[train_idx], y[train_idx]
X_val, y_val_phys = X[val_idx], y[val_idx]
X_cal, y_cal_phys = X[cal_idx], y[cal_idx]
X_test, y_test_phys = X[test_idx], y[test_idx]

In [6]:
label_stats_path = DATA_PROCESSED / f"{dataset_id}_label_stats_train_only.npz"

y_params = np.load(label_stats_path)
y_mean = y_params["mean"]
y_std = y_params["std"]

y_train_std = (y_train_phys - y_mean) / y_std
y_val_std = (y_val_phys - y_mean) / y_std
y_cal_std = (y_cal_phys - y_mean) / y_std
y_test_std = (y_test_phys - y_mean) / y_std

## Datasets and Dataloaders

In [7]:
from torch.utils.data import DataLoader
from src.models.dataset import ArrayRegressionDataset

batch_size = 32

train_dataset = ArrayRegressionDataset(X_train, y_train_std)
val_dataset = ArrayRegressionDataset(X_val, y_val_std)
cal_dataset = ArrayRegressionDataset(X_cal, y_cal_std)
test_dataset = ArrayRegressionDataset(X_test, y_test_std)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
)

cal_loader = DataLoader(
    cal_dataset,
    batch_size=batch_size,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,

)

## Load the checkpoint

In [8]:
import torch
from src.models.network import SimpleCNN


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_file_name = "bbh_processed_4s_seobnrv4opt_snr10-25_n4500_checkpoint.pt"

checkpoint_path = CHECKPOINTS_DIR / checkpoint_file_name

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

model_config = checkpoint["model_config"]

In [25]:
import torch
from src.models.network import SimpleCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_file_name = "bbh_processed_4s_seobnrv4opt_snr10-25_n4500_checkpoint.pt"
checkpoint_path = CHECKPOINTS_DIR / checkpoint_file_name

checkpoint = torch.load(checkpoint_path, map_location=device)

model_config = checkpoint["model_config"]

model = SimpleCNN(
    n_detectors=model_config["n_detectors"],
    n_outputs=model_config["n_outputs"],
    embedding_dim=model_config["embedding_dim"],
    dropout_conv=model_config["dropout_conv"],
    dropout_dense=model_config["dropout_dense"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded epoch:", checkpoint["epoch"])
print("Loaded best val loss:", checkpoint["best_val_loss"])

Loaded epoch: 98
Loaded best val loss: 0.513808821439743


This function uses the model for the prediction inference

In [27]:
def predict_set(model, dataloader, device):
    model.eval()
    pred = []
    true = []
    emb = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)

            pred_batch, emb_batch = model(X_batch, return_embedding=True)

            pred.append(pred_batch.cpu().numpy())
            true.append(y_batch.numpy())
            emb.append(emb_batch.cpu().numpy())
            
    return np.concatenate(pred, axis=0), np.concatenate(true, axis=0), np.concatenate(emb, axis=0)

In [28]:
#Standardized predictions (y_std where used in the dataloader)
pred_train, y_train, emb_train = predict_set(model, train_loader, device)
pred_val, y_val, emb_val = predict_set(model, val_loader, device)
pred_cal, y_cal, emb_cal = predict_set(model, cal_loader, device)
pred_test, y_test, emb_test = predict_set(model, test_loader, device)



In [ ]:
def mse(pred, y):
    if len(y)!=len(pred):
        raise ("y and pred must have the same length")
    
    return (np.sum( (pred - y)**2, axis=0 ) / len(y))


def mae(pred, y):
    if len(y)!=len(pred):
        raise ("y and pred must have the same length")
    
    return (np.sum( np.abs(pred - y), axis=0 ) / len(y))


def bias(pred, y):
    if len(y)!=len(pred):
        raise ("y and pred must have the same length")
    
    return (np.sum( (pred - y), axis=0 ) / len(y))


def median_mae(pred, y):
    if len(y)!=len(pred):
        raise ("y and pred must have the same length")
    
    return np.median( np.abs(pred - y), axis=0)


def R2(pred, y):
    """
    El coeficiente R2 mide qué fracción de la varianza del target explica el modelo.
        - R² = 1 -> predicción perfecta
        - R² = 0 -> igual que predecir siempre la media del target
        - R² < 0 -> peor que predecir siempre la media
    """
    if len(y)!=len(pred):
        raise ("y and pred must have the same length")
    
    ss_res = np.sum((y - pred) ** 2, axis=0)
    ss_tot = np.sum((y - np.mean(y, axis=0)) ** 2, axis=0)
    r2 = 1.0 - ss_res / ss_tot

    return r2


In [31]:
mse_train = mse(pred_train, y_train)
mse_val = mse(pred_val, y_val)

rmse_train = np.sqrt(mse_train)

mae_train = mae(pred_train, y_train)

bias_train = bias(pred_train, y_train)

median_mae_train = median_mae(pred_train, y_train)

r2_train = R2(pred_train, y_train)

residual_std_train = pred_train - y_train
residual_std_val = pred_val - y_val
residual_std_cal = pred_cal - y_cal
residual_std_test = pred_test - y_test

abs_residual_std_train = np.abs(residual_std_train)
abs_residual_std_val = np.abs(residual_std_val)
abs_residual_std_cal = np.abs(residual_std_cal)
abs_residual_std_test = np.abs(residual_std_test)

print(mse_train)
print(rmse_train)
print(mae_train)
print(bias_train)
print(median_mae_train)
print(r2_train)

[0.30435142 0.27968857 0.8296237 ]
[0.55168056 0.5288559  0.9108368 ]
[0.4245055  0.42935112 0.7536654 ]
[-0.01757807 -0.0238964   0.04207357]
[0.33581963 0.3744296  0.6878182 ]
[0.6956485  0.72031164 0.17037624]


In [32]:
import pandas as pd
from src.models.evaluate import regression_metrics

metrics_train_std = regression_metrics(y_train, pred_train, label_names, "train_std")
metrics_val_std   = regression_metrics(y_val, pred_val, label_names, "val_std")
metrics_cal_std   = regression_metrics(y_cal, pred_cal, label_names, "cal_std")
metrics_test_std  = regression_metrics(y_test, pred_test, label_names, "test_std")

metrics_all_std = pd.concat(
    [metrics_train_std, metrics_val_std, metrics_cal_std, metrics_test_std],
    ignore_index=True
)

metrics_all_std

,split,label,MSE,RMSE,MAE,bias,median_abs_error,residual_std,R2
0,train_std,chirp_mass,0.304351,0.551681,0.424506,-0.017578,0.335820,0.551401,0.695648
1,train_std,total_mass,0.279689,0.528856,0.429351,-0.023896,0.374430,0.528315,0.720312
2,train_std,chi_eff,0.829624,0.910837,0.753665,0.042074,0.687818,0.909865,0.170376
3,val_std,chirp_mass,0.399915,0.632388,0.487838,0.021110,0.412736,0.632035,0.604641
4,val_std,total_mass,0.359042,0.599201,0.491980,0.021508,0.452541,0.598815,0.645375
5,val_std,chi_eff,0.782469,0.884573,0.723397,0.090520,0.608766,0.879929,0.187684
6,cal_std,chirp_mass,0.424031,0.651177,0.497136,0.010041,0.376244,0.651100,0.591320
7,cal_std,total_mass,0.386476,0.621672,0.502567,-0.017962,0.429136,0.621413,0.626739
8,cal_std,chi_eff,0.875702,0.935790,0.772089,0.124701,0.703322,0.927444,0.159036
9,test_std,chirp_mass,0.486036,0.697163,0.531758,0.014209,0.415155,0.697018,0.528750


In physical space

In [33]:
y_test_phys = y_test * y_std + y_mean
pred_test_phys = pred_test * y_std + y_mean

In [34]:
metrics_test_phys = regression_metrics(
    y_true=y_test_phys,
    y_pred=pred_test_phys,
    label_names=label_names,
    split_name="test_phys"
)

metrics_test_phys

,split,label,MSE,RMSE,MAE,bias,median_abs_error,residual_std,R2
0,test_phys,chirp_mass,129.435974,11.376993,8.677752,0.231874,6.774901,11.374628,0.528750
1,test_phys,total_mass,480.950256,21.930578,17.588696,0.475389,14.880852,21.925428,0.589690
2,test_phys,chi_eff,0.156081,0.395071,0.327469,0.039567,0.300857,0.393085,0.207371
